In [0]:
%pip install langgraph==0.2.65 langchain==0.3.16 langchain-core==0.3.32 pydantic==2.10.6 sqlparse==0.5.3 openai==1.59.7 --quiet
dbutils.library.restartPython()

## 📦 Installation & Imports

Installing required dependencies for LangGraph workflow and Claude integration.

In [0]:
import os
from typing import List, Optional, Literal, TypedDict, Annotated, Dict, Any
from datetime import datetime, timedelta
import pandas as pd
import json
import re
from pydantic import BaseModel, Field, validator
from langgraph.graph import StateGraph, END
from langchain_core.output_parsers import PydanticOutputParser
import sqlparse
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import openai

print("✅ All libraries imported successfully")

## 🔧 Configuration

Configure source tables, target schema, and LLM settings.

   
### ⚠️ **IMPORTANT: Configuration Required**

**Before running this notebook, you must update the following:**

#### 1. Table Configuration (Cell 6 - Input Configs)
Replace the placeholder values with your actual table information:

```python
SOURCE_CATALOG = "your_catalog_name"     # TODO: Your source catalog
SOURCE_SCHEMA = "your_schema_name"       # TODO: Your source schema  
SOURCE_TABLE = "your_table_name"         # TODO: Your source table
TARGET_CATALOG = "your_target_catalog"   # TODO: Target catalog for KPI views
TARGET_SCHEMA = "your_target_schema"     # TODO: Target schema for KPI views
```


**After updating:**
1. Run Cell 6 (Input Configs)
2. Run Cell 7 (System Configuration)  
3. Run Cell 13 (Initialize Claude LLM)
4. Then run Cell 28 (Test Run)

In [0]:
SOURCE_CATALOG = "your_catalog_name"
SOURCE_SCHEMA = "your_schema_name"
SOURCE_TABLE = "your_table_name"
TARGET_CATALOG = "your_target_catalog"
TARGET_SCHEMA = "your_target_schema"

In [0]:
# AI Gateway Configuration (Tiger Analytics)
# Option 1: Use Databricks Secrets (RECOMMENDED for production)
# API_KEY = dbutils.secrets.get(scope="your-scope", key="ai-gateway-key")

# Option 2: Direct API Key (for testing)
API_KEY = "sk-YFxU38F33HyLN_P-KexwPw"

BASE_URL = "https://api.ai-gateway.tigeranalytics.com"
MODEL_NAME = "claude-opus-4.6"

# Model Configuration
TEMPERATURE = 0.1
MAX_TOKENS = 15000

# Full table paths
SOURCE_TABLE_FULL = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{SOURCE_TABLE}"
VIEW_PREFIX = "vw_kpi"

# Sampling configuration for analysis
SAMPLE_ROWS = 100

# Execution mode
DRY_RUN = False  # Set to True to skip actual view creation
VALIDATE_SQL = True  # Set to True to validate SQL before execution

print(f"""
╔════════════════════════════════════════════╗
║      Configuration Loaded                  ║
╠════════════════════════════════════════════╣
║ 📊 Source: {SOURCE_TABLE_FULL:<28} ║
║ 🎯 Target: {TARGET_CATALOG}.{TARGET_SCHEMA:<24} ║
║ 🔍 Sample: {SAMPLE_ROWS} rows{' '*21} ║
║ 🤖 Model:  {MODEL_NAME:<28} ║
║ 🧪 Dry Run: {DRY_RUN!s:<27} ║
╚════════════════════════════════════════════╝
""")

## 📋 Pydantic Models

Defining structured data models for KPI definitions and validation.

In [0]:
class KPIDefinition(BaseModel):
    """Individual KPI Definition with validation"""
    
    kpi_id: str = Field(
        description="Unique identifier for the KPI (e.g., KPI_001, KPI_002)",
        pattern=r'^KPI_\d{3}$'
    )
    
    kpi_name: str = Field(
        description="Short, descriptive name for the KPI (max 100 chars)",
        max_length=100
    )
    
    kpi_description: str = Field(
        description="Business description of the KPI in 3-4 lines explaining what it measures and why it matters",
        min_length=50,
        max_length=500
    )
    
    kpi_category: Literal[
        "Revenue", "Volume", "Price", "Change_Detection", "Geographical", 
        "Temporal", "Product", "Customer", "Operational", "Quality"
    ] = Field(
        description="Category of the KPI for organization"
    )
    
    sql_view_name: str = Field(
        description="Name for the view (e.g., vw_kpi_daily_revenue_change)",
        pattern=r'^vw_kpi_[a-z0-9_]+$'
    )
    
    sql_command: str = Field(
        description="Complete CREATE OR REPLACE VIEW SQL statement to generate this KPI"
    )
    
    grain_level: str = Field(
        description="Aggregation level (e.g., 'Location, Date', 'Product Category', 'Daily')",
        max_length=200
    )
    
    expected_output_columns: List[str] = Field(
        description="List of column names that will be in the view output",
        min_items=1
    )
    
    business_priority: Literal["Critical", "High", "Medium", "Low"] = Field(
        description="Business importance of this KPI"
    )
    
    update_frequency: Literal["Real-time", "Hourly", "Daily", "Weekly", "Monthly"] = Field(
        description="How often this KPI should be refreshed"
    )
    
    @validator('sql_command')
    def validate_sql(cls, v):
        """Basic SQL validation"""
        v = v.strip()
        upper_v = v.upper()
        if not (upper_v.startswith('CREATE OR REPLACE VIEW') or upper_v.startswith('CREATE VIEW')):
            raise ValueError("SQL must start with CREATE OR REPLACE VIEW or CREATE VIEW")
        if 'SELECT' not in upper_v:
            raise ValueError("SQL must contain SELECT statement")
        return v
    
    @validator('sql_view_name')
    def validate_view_name(cls, v):
        """Validate view name format"""
        if len(v) > 64:
            raise ValueError("View name must be under 64 characters")
        if not v.startswith('vw_kpi_'):
            raise ValueError("View name must start with 'vw_kpi_'")
        return v.lower()


class KPIGenerationOutput(BaseModel):
    """Complete output of KPI generation process"""
    
    source_table: str = Field(description="Full source table name (catalog.schema.table)")
    domain: str = Field(description="Identified business domain")
    domain_confidence: Literal["High", "Medium", "Low"] = Field(
        description="Confidence level in domain identification"
    )
    
    kpis: List[KPIDefinition] = Field(
        description="List of generated KPIs (exactly 10)",
        min_items=10,
        max_items=10
    )
    
    target_catalog: str = Field(description="Target catalog for views")
    target_schema: str = Field(description="Target schema for views")
    generation_timestamp: str = Field(
        default_factory=lambda: datetime.now().isoformat()
    )
    
    total_kpis: int = Field(default=10)
    
    table_statistics: Dict[str, Any] = Field(
        default_factory=dict,
        description="Statistics about the source table"
    )
    
    @validator('kpis')
    def validate_kpi_count(cls, v):
        """Ensure exactly 10 KPIs"""
        if len(v) != 10:
            raise ValueError(f"Must generate exactly 10 KPIs, got {len(v)}")
        
        # Check for duplicate KPI IDs
        kpi_ids = [kpi.kpi_id for kpi in v]
        if len(kpi_ids) != len(set(kpi_ids)):
            raise ValueError("Duplicate KPI IDs found")
        
        # Check for duplicate view names
        view_names = [kpi.sql_view_name for kpi in v]
        if len(view_names) != len(set(view_names)):
            raise ValueError("Duplicate view names found")
        
        return v


print("✅ Pydantic models defined successfully")

## 🔄 Graph State Definition

Defining the state object that flows through the LangGraph workflow.

In [0]:
class GraphState(TypedDict):
    """State object that flows through the LangGraph workflow"""
    
    # Input parameters
    source_table: str
    target_catalog: str
    target_schema: str
    
    # Table information
    table_schema: Optional[Dict[str, str]]
    sample_data: Optional[str]
    table_stats: Optional[Dict[str, Any]]
    column_info_summary: Optional[str]
    
    # Domain analysis
    domain: Optional[str]
    domain_confidence: Optional[str]
    domain_analysis: Optional[str]
    
    # KPI generation
    kpis: Optional[List[Dict[str, Any]]]
    kpi_output: Optional[Dict[str, Any]]
    
    # Validation results
    validation_results: Optional[Dict[str, Any]]
    validation_passed: Optional[bool]
    
    # Execution results
    execution_results: Optional[List[Dict[str, Any]]]
    views_created: Optional[List[str]]
    
    # Final output
    final_dataframe: Optional[pd.DataFrame]
    
    # Workflow control
    current_step: str
    errors: List[str]
    warnings: List[str]
    retry_count: int
    
    # Metadata
    start_time: Optional[str]
    end_time: Optional[str]


print("✅ Graph state defined successfully")

## 🤖 LLM Initialization

Initializing Claude model for KPI generation and domain analysis.

In [0]:
import openai

# Initialize the OpenAI client with AI Gateway
client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("✅ OpenAI client initialized successfully")
print(f"   Gateway: {BASE_URL}")
print(f"   Model: {MODEL_NAME}")
print(f"   Temperature: {TEMPERATURE}")

## 📝 System Prompts

Defining AI prompts for domain identification and KPI generation.

In [0]:
# Domain Identification Prompt
DOMAIN_IDENTIFICATION_PROMPT = """You are a business intelligence expert analyzing a data table to identify its business domain.

Given the following table information:

**Table Name:** {table_name}

**Schema:**
{schema}

**Sample Data (first few rows):**
{sample_data}

**Table Statistics:**
{table_stats}

Analyze this table and identify:
1. The primary business domain (e.g., Retail, Healthcare, Finance, Manufacturing, etc.)
2. Your confidence level (High/Medium/Low)
3. A brief analysis explaining your reasoning

Provide your response in this exact JSON format:
{{
    "domain": "<domain name>",
    "confidence": "<High|Medium|Low>",
    "analysis": "<2-3 sentences explaining the reasoning>"
}}
"""

# KPI Generation Prompt
KPI_GENERATION_PROMPT = """You are a KPI generation expert. Generate exactly 10 high-impact KPIs for the following business domain.

**Business Domain:** {domain}
**Domain Analysis:** {domain_analysis}

**Source Table:** {table_name}

**Table Schema:**
{schema}

**Sample Data:**
{sample_data}

**Target Location:** {target_catalog}.{target_schema}

**Requirements:**
1. Generate EXACTLY 10 KPIs
2. Each KPI must include a complete CREATE OR REPLACE VIEW statement
3. KPIs should focus on change detection, trends, and business insights
4. Use appropriate aggregations (daily, weekly, monthly)
5. Include at least 2 KPIs focused on change detection (e.g., week-over-week, month-over-month)
6. Ensure SQL is valid Databricks SQL syntax
7. View names must start with 'vw_kpi_' and be lowercase with underscores
8. Each KPI must have a unique ID in format KPI_001, KPI_002, etc.

**Output Format:**
Return a JSON object matching this structure:

{format_instructions}
"""

print("✅ System prompts defined successfully")

## 🛠️ Helper Functions

Utility functions for table introspection and SQL validation.

In [0]:
def get_table_info(table_name: str, sample_rows: int = 100) -> Dict[str, Any]:
    """Extract comprehensive information about a Unity Catalog table"""
    try:
        # Read table schema
        df = spark.table(table_name)
        
        # Get schema information
        schema_dict = {field.name: str(field.dataType) for field in df.schema.fields}
        
        # Get sample data
        sample_df = df.limit(sample_rows).toPandas()
        
        # Get table statistics
        row_count = df.count()
        col_count = len(df.columns)
        
        # Format sample data as string
        sample_data_str = sample_df.head(10).to_string(index=False)
        
        # Create column info summary
        column_info = []
        for col_name, col_type in schema_dict.items():
            column_info.append(f"  - {col_name}: {col_type}")
        column_info_summary = "\n".join(column_info)
        
        table_stats = {
            "row_count": row_count,
            "column_count": col_count,
            "columns": list(schema_dict.keys())
        }
        
        return {
            "schema": schema_dict,
            "sample_data": sample_data_str,
            "table_stats": table_stats,
            "column_info_summary": column_info_summary,
            "success": True
        }
    except Exception as e:
        return {
            "error": str(e),
            "success": False
        }


def validate_sql_syntax(sql: str) -> Dict[str, Any]:
    """Validate SQL syntax using sqlparse"""
    try:
        # Parse the SQL
        parsed = sqlparse.parse(sql)
        
        if not parsed:
            return {
                "valid": False,
                "error": "Could not parse SQL statement"
            }
        
        # Check for basic structure
        sql_upper = sql.upper().strip()
        if not ('CREATE' in sql_upper and 'VIEW' in sql_upper and 'SELECT' in sql_upper):
            return {
                "valid": False,
                "error": "SQL must be a CREATE VIEW statement with SELECT"
            }
        
        return {
            "valid": True,
            "formatted_sql": sqlparse.format(sql, reindent=True, keyword_case='upper')
        }
    except Exception as e:
        return {
            "valid": False,
            "error": str(e)
        }


print("✅ Helper functions defined successfully")

## 🔄 Workflow Node Functions

LangGraph workflow nodes for the KPI generation pipeline.

In [0]:
def extract_table_info_node(state: GraphState) -> GraphState:
    """Node 1: Extract table information from Unity Catalog"""
    print("\n" + "="*60)
    print("📊 STEP 1: Extracting Table Information")
    print("="*60)
    
    table_name = state["source_table"]
    print(f"Analyzing table: {table_name}")
    
    # Get table information
    table_info = get_table_info(table_name, SAMPLE_ROWS)
    
    if not table_info["success"]:
        state["errors"].append(f"Failed to extract table info: {table_info.get('error')}")
        print(f"❌ Error: {table_info.get('error')}")
        return state
    
    # Update state
    state["table_schema"] = table_info["schema"]
    state["sample_data"] = table_info["sample_data"]
    state["table_stats"] = table_info["table_stats"]
    state["column_info_summary"] = table_info["column_info_summary"]
    state["current_step"] = "extract_table_info"
    
    print(f"✅ Successfully extracted info:")
    print(f"   - Rows: {table_info['table_stats']['row_count']:,}")
    print(f"   - Columns: {table_info['table_stats']['column_count']}")
    print(f"   - Sample: {SAMPLE_ROWS} rows retrieved")
    
    return state

print("✅ extract_table_info_node defined")

In [0]:
def identify_domain_node(state: GraphState) -> GraphState:
    """Node 2: Identify business domain using LLM"""
    print("\n" + "="*60)
    print("🔍 STEP 2: Identifying Business Domain")
    print("="*60)
    
    # Prepare prompt
    prompt = DOMAIN_IDENTIFICATION_PROMPT.format(
        table_name=state["source_table"],
        schema=state["column_info_summary"],
        sample_data=state["sample_data"][:1000],  # Limit sample data length
        table_stats=json.dumps(state["table_stats"], indent=2)
    )
    
    try:
        # Call LLM via OpenAI client
        print("Calling Claude for domain identification...")
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )
        
        # Parse JSON response
        response_text = response.choices[0].message.content
        
        # Extract JSON from response
        json_match = re.search(r'\{[^}]+\}', response_text, re.DOTALL)
        if json_match:
            domain_info = json.loads(json_match.group())
        else:
            domain_info = json.loads(response_text)
        
        # Update state
        state["domain"] = domain_info["domain"]
        state["domain_confidence"] = domain_info["confidence"]
        state["domain_analysis"] = domain_info["analysis"]
        state["current_step"] = "identify_domain"
        
        print(f"✅ Domain identified:")
        print(f"   - Domain: {state['domain']}")
        print(f"   - Confidence: {state['domain_confidence']}")
        print(f"   - Analysis: {state['domain_analysis'][:100]}...")
        
    except Exception as e:
        state["errors"].append(f"Failed to identify domain: {str(e)}")
        print(f"❌ Error: {str(e)}")
    
    return state

print("✅ identify_domain_node defined")

In [0]:
def generate_kpis_node(state: GraphState) -> GraphState:
    """Node 3: Generate 10 KPIs using LLM"""
    print("\n" + "="*60)
    print("🎯 STEP 3: Generating KPIs")
    print("="*60)
    
    # Create parser
    parser = PydanticOutputParser(pydantic_object=KPIGenerationOutput)
    
    # Prepare prompt
    prompt = KPI_GENERATION_PROMPT.format(
        domain=state["domain"],
        domain_analysis=state["domain_analysis"],
        table_name=state["source_table"],
        schema=state["column_info_summary"],
        sample_data=state["sample_data"][:1500],
        target_catalog=state["target_catalog"],
        target_schema=state["target_schema"],
        format_instructions=parser.get_format_instructions()
    )
    
    try:
        # Call LLM via OpenAI client
        print("Calling Claude for KPI generation...")
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )
        
        # Parse response
        response_text = response.choices[0].message.content
        
        # Try to extract JSON
        try:
            # Try direct parsing first
            kpi_output = parser.parse(response_text)
        except:
            # Try to extract JSON block
            json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if json_match:
                kpi_output = parser.parse(json_match.group())
            else:
                raise ValueError("Could not extract JSON from response")
        
        # Convert to dict
        state["kpi_output"] = kpi_output.dict()
        state["kpis"] = [kpi.dict() for kpi in kpi_output.kpis]
        state["current_step"] = "generate_kpis"
        
        print(f"✅ Successfully generated {len(state['kpis'])} KPIs:")
        for i, kpi in enumerate(state["kpis"], 1):
            print(f"   {i}. {kpi['kpi_name']} ({kpi['kpi_category']})")
        
    except Exception as e:
        state["errors"].append(f"Failed to generate KPIs: {str(e)}")
        print(f"❌ Error: {str(e)}")
    
    return state

print("✅ generate_kpis_node defined")

In [0]:
def validate_kpis_node(state: GraphState) -> GraphState:
    """Node 4: Validate SQL for each KPI"""
    print("\n" + "="*60)
    print("✓ STEP 4: Validating KPI SQL")
    print("="*60)
    
    validation_results = []
    all_valid = True
    
    for kpi in state["kpis"]:
        kpi_id = kpi["kpi_id"]
        sql = kpi["sql_command"]
        
        print(f"\nValidating {kpi_id}: {kpi['kpi_name']}")
        
        # Validate SQL syntax
        validation = validate_sql_syntax(sql)
        
        validation_results.append({
            "kpi_id": kpi_id,
            "kpi_name": kpi["kpi_name"],
            "valid": validation["valid"],
            "error": validation.get("error", None)
        })
        
        if validation["valid"]:
            print(f"   ✅ Valid SQL")
        else:
            print(f"   ❌ Invalid: {validation['error']}")
            all_valid = False
            state["warnings"].append(f"{kpi_id}: {validation['error']}")
    
    state["validation_results"] = {"results": validation_results}
    state["validation_passed"] = all_valid
    state["current_step"] = "validate_kpis"
    
    if all_valid:
        print(f"\n✅ All KPIs passed validation")
    else:
        print(f"\n⚠️  Some KPIs have validation warnings")
    
    return state

print("✅ validate_kpis_node defined")

In [0]:
def create_views_node(state: GraphState) -> GraphState:
    """Node 5: Create views in Unity Catalog"""
    print("\n" + "="*60)
    print("🏗️  STEP 5: Creating Views in Unity Catalog")
    print("="*60)
    
    execution_results = []
    views_created = []
    
    if DRY_RUN:
        print("\n⚠️  DRY RUN MODE - Views will not be created\n")
    
    for kpi in state["kpis"]:
        kpi_id = kpi["kpi_id"]
        view_name = f"{state['target_catalog']}.{state['target_schema']}.{kpi['sql_view_name']}"
        sql = kpi["sql_command"]
        
        print(f"\n{kpi_id}: {kpi['kpi_name']}")
        print(f"View: {view_name}")
        
        if DRY_RUN:
            print(f"   🔍 SQL Preview:")
            print(f"   {sql[:200]}...")
            execution_results.append({
                "kpi_id": kpi_id,
                "view_name": view_name,
                "status": "skipped (dry run)",
                "error": None
            })
            continue
        
        try:
            # Execute CREATE VIEW
            spark.sql(sql)
            print(f"   ✅ View created successfully")
            
            views_created.append(view_name)
            execution_results.append({
                "kpi_id": kpi_id,
                "view_name": view_name,
                "status": "success",
                "error": None
            })
            
        except Exception as e:
            print(f"   ❌ Failed: {str(e)[:100]}")
            execution_results.append({
                "kpi_id": kpi_id,
                "view_name": view_name,
                "status": "failed",
                "error": str(e)
            })
            state["errors"].append(f"{kpi_id}: {str(e)}")
    
    state["execution_results"] = execution_results
    state["views_created"] = views_created
    state["current_step"] = "create_views"
    
    if not DRY_RUN:
        print(f"\n✅ Created {len(views_created)} views successfully")
    
    return state

print("✅ create_views_node defined")

## 🕸️ Build LangGraph Workflow

Constructing the complete KPI generation workflow.

In [0]:
def build_workflow():
    """Build the complete LangGraph workflow"""
    
    # Create workflow
    workflow = StateGraph(GraphState)
    
    # Add nodes
    workflow.add_node("extract_table_info", extract_table_info_node)
    workflow.add_node("identify_domain", identify_domain_node)
    workflow.add_node("generate_kpis", generate_kpis_node)
    workflow.add_node("validate_kpis", validate_kpis_node)
    workflow.add_node("create_views", create_views_node)
    
    # Define edges
    workflow.set_entry_point("extract_table_info")
    workflow.add_edge("extract_table_info", "identify_domain")
    workflow.add_edge("identify_domain", "generate_kpis")
    workflow.add_edge("generate_kpis", "validate_kpis")
    workflow.add_edge("validate_kpis", "create_views")
    workflow.add_edge("create_views", END)
    
    # Compile
    app = workflow.compile()
    
    return app

# Build the workflow
workflow_app = build_workflow()

print("✅ LangGraph workflow compiled successfully")
print("\nWorkflow Steps:")
print("  1. Extract Table Info")
print("  2. Identify Domain")
print("  3. Generate KPIs")
print("  4. Validate KPIs")
print("  5. Create Views")

## 🚀 Execution Function

Main function to run the complete KPI generation workflow.

In [0]:
def run_kpi_generation(
    source_table: str,
    target_catalog: str,
    target_schema: str
) -> Dict[str, Any]:
    """Run the complete KPI generation workflow"""
    
    print("\n" + "="*70)
    print("🎯 AUTOMATED KPI GENERATION SYSTEM")
    print("="*70)
    print(f"\nSource: {source_table}")
    print(f"Target: {target_catalog}.{target_schema}")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("\n" + "="*70)
    
    # Initialize state
    initial_state = {
        "source_table": source_table,
        "target_catalog": target_catalog,
        "target_schema": target_schema,
        "table_schema": None,
        "sample_data": None,
        "table_stats": None,
        "column_info_summary": None,
        "domain": None,
        "domain_confidence": None,
        "domain_analysis": None,
        "kpis": None,
        "kpi_output": None,
        "validation_results": None,
        "validation_passed": None,
        "execution_results": None,
        "views_created": None,
        "final_dataframe": None,
        "current_step": "start",
        "errors": [],
        "warnings": [],
        "retry_count": 0,
        "start_time": datetime.now().isoformat(),
        "end_time": None
    }
    
    try:
        # Run workflow
        final_state = workflow_app.invoke(initial_state)
        
        # Update end time
        final_state["end_time"] = datetime.now().isoformat()
        
        # Print summary
        print("\n" + "="*70)
        print("📊 EXECUTION SUMMARY")
        print("="*70)
        print(f"\n✅ Workflow completed successfully")
        print(f"\n📈 Results:")
        print(f"   - Domain: {final_state.get('domain', 'N/A')}")
        print(f"   - KPIs Generated: {len(final_state.get('kpis', []))}")
        print(f"   - Views Created: {len(final_state.get('views_created', []))}")
        
        if final_state.get('errors'):
            print(f"\n⚠️  Errors: {len(final_state['errors'])}")
            for err in final_state['errors']:
                print(f"   - {err}")
        
        if final_state.get('warnings'):
            print(f"\n⚠️  Warnings: {len(final_state['warnings'])}")
            for warn in final_state['warnings']:
                print(f"   - {warn}")
        
        # Create summary DataFrame
        if final_state.get('kpis'):
            kpi_summary = []
            for kpi in final_state['kpis']:
                kpi_summary.append({
                    'KPI_ID': kpi['kpi_id'],
                    'KPI_Name': kpi['kpi_name'],
                    'Category': kpi['kpi_category'],
                    'Priority': kpi['business_priority'],
                    'Update_Frequency': kpi['update_frequency'],
                    'View_Name': f"{target_catalog}.{target_schema}.{kpi['sql_view_name']}"
                })
            final_state['final_dataframe'] = pd.DataFrame(kpi_summary)
        
        print("\n" + "="*70)
        
        return final_state
        
    except Exception as e:
        print(f"\n❌ Workflow failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            "errors": [str(e)],
            "end_time": datetime.now().isoformat()
        }

print("✅ Main execution function defined")

## 🧪 Test Execution

Ready to test! Update the configuration and run the workflow.

In [0]:
# ⚠️ IMPORTANT: Update the configuration in Cell 6 before running this!
# Make sure to set:
# - SOURCE_CATALOG, SOURCE_SCHEMA, SOURCE_TABLE to your actual table
# - TARGET_CATALOG, TARGET_SCHEMA to where you want views created
# - Set DRY_RUN = True for testing without creating views

# Uncomment the lines below to run:
result = run_kpi_generation(
    source_table=SOURCE_TABLE_FULL,
    target_catalog=TARGET_CATALOG,
    target_schema=TARGET_SCHEMA
)
final_df = spark.createDataFrame(result['kpis'])
display(final_df)

# # Display results
# if result.get('final_dataframe') is not None:
#     display(result['final_dataframe'])

print("⚠️  Test execution cell ready")
print("\nTo run:")
print("  1. Update configuration in Cell 6")
print("  2. Uncomment the code in this cell")
print("  3. Run this cell")

In [0]:
from IPython.display import display, Image
import base64

# Define Mermaid diagram for the workflow
mermaid_diagram = """
graph TD
    Start([Start]) --> A[📊 Extract Table Info]
    A --> B[🔍 Identify Domain]
    B --> C[🎯 Generate KPIs]
    C --> D[✓ Validate KPIs]
    D --> E[🏗️ Create Views]
    E --> End([End])
    
    style Start fill:#e1f5e1
    style End fill:#ffe1e1
    style A fill:#e3f2fd
    style B fill:#f3e5f5
    style C fill:#fff3e0
    style D fill:#e8f5e9
    style E fill:#fce4ec
"""

print("🕸️ KPI Generation Workflow Graph\n")
print("="*70)
print("\nWorkflow Steps:")
print("  1. 📊 Extract Table Info - Analyze source table schema and data")
print("  2. 🔍 Identify Domain - AI-powered business domain detection")
print("  3. 🎯 Generate KPIs - Create 10 tailored KPIs using Claude")
print("  4. ✓ Validate KPIs - Check SQL syntax for all generated queries")
print("  5. 🏗️ Create Views - Deploy KPI views to Unity Catalog")
print("\n" + "="*70)

# Try to render with Mermaid if available
try:
    from IPython.display import HTML
    
    html_content = f"""
    <div class="mermaid">
    {mermaid_diagram}
    </div>
    <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
    <script>
        mermaid.initialize({{startOnLoad:true}});
    </script>
    """
    
    display(HTML(html_content))
    print("\n✅ Workflow graph rendered above")
    
except Exception as e:
    print(f"\n⚠️ Mermaid rendering not available in this environment")
    print("\nWorkflow Flow:")
    print("  Start → Extract Table Info → Identify Domain → Generate KPIs → Validate KPIs → Create Views → End")

## ✅ Notebook Complete!

---

### 🎉 All Components Implemented

The notebook is fully functional with:
* ✅ Pydantic models for structured KPI definitions
* ✅ LangGraph workflow with 5 nodes
* ✅ Claude integration for AI-powered KPI generation
* ✅ SQL validation and view creation
* ✅ Comprehensive error handling

---

### 🚀 How to Use

#### Step 1: Configure Your Environment

Update **Cell 6 (System Configuration)** with your actual values:

```python
# Update these values:
SOURCE_CATALOG = "your_catalog"      # Your Unity Catalog name
SOURCE_SCHEMA = "silver"             # Schema containing source table
SOURCE_TABLE = "retail_transactions" # Your actual table name

TARGET_CATALOG = "your_catalog"      # Where to create KPI views
TARGET_SCHEMA = "gold"               # Target schema for views

# For testing without creating views:
DRY_RUN = True  # Set to False to actually create views
```

#### Step 2: Run All Cells

Run all cells from top to bottom to initialize the workflow.

#### Step 3: Execute the Workflow

Uncomment and run **Cell 28 (Test Run)** or create a new cell:

```python
result = run_kpi_generation(
    source_table=SOURCE_TABLE_FULL,
    target_catalog=TARGET_CATALOG,
    target_schema=TARGET_SCHEMA
)

# Display the generated KPIs
if result.get('final_dataframe') is not None:
    display(result['final_dataframe'])
```

---

### 📊 What the Workflow Does

1. **Extracts Table Info** - Analyzes your source table schema and data
2. **Identifies Domain** - Uses AI to determine business domain (Retail, Finance, etc.)
3. **Generates 10 KPIs** - Creates tailored KPIs based on your data
4. **Validates SQL** - Checks all generated SQL statements
5. **Creates Views** - Deploys KPI views to Unity Catalog (unless DRY_RUN=True)

---

### 💡 Example Tables to Try

* **Retail:** Transaction tables, customer purchases, inventory
* **Finance:** Account balances, transactions, investments
* **Healthcare:** Patient records, appointments, treatments
* **Manufacturing:** Production data, quality metrics, inventory

---

### 🔍 Expected Output

The workflow generates a DataFrame with 10 KPIs including:
* KPI ID and Name
* Category (Revenue, Volume, Change Detection, etc.)
* Business Priority
* Update Frequency
* Full view name in Unity Catalog

Each KPI includes a complete CREATE VIEW SQL statement.